In [5]:
with open(csv_path, mode='r', encoding='utf-8') as file:
    first_line = file.readline()
    print("Première ligne :", first_line)

Première ligne : No disposition,Date mutation,Valeur fonciere,Adresse_ventes,Section,No plan,1er lot,Surface Carrez du 1er lot,Nombre de lots,Type local,Surface reelle bati,Nombre pieces principales,Nature culture,Nature culture speciale,Surface terrain,N_ventes,No disposition



In [ ]:
import mysql.connector
import csv
from datetime import datetime

# Fonction pour convertir les dates CSV (JJ/MM/AAAA) en format MySQL (AAAA-MM-JJ)
def convert_date(date_str):
    if not date_str or not date_str.strip():
        return None  # si la cellule est vide
    try:
        return datetime.strptime(date_str, "%d/%m/%Y").date().isoformat()
    except ValueError:
        return None  # en cas d'erreur, on met NULL
        
def to_int_or_none(value):
    if not value or value.strip() == "":
        return None
    return int(float(value))  # float() pour gérer "164.87" puis int() si tu veux un entier

def to_float_or_none(value):
    if not value or value.strip() == "":
        return None
    return float(value.replace(",", "."))  # au cas où ton CSV a des virgules


# Connexion à MySQL
conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password=".....",
    database="valeurs_foncieres"
)
cursor = conn.cursor()

# Chemin du fichier CSV
csv_path = r"........"

with open(csv_path, mode='r', encoding='utf-8') as file:
    reader = csv.DictReader(file, delimiter=',')
    
    # Supprimer le BOM si présent
    reader.fieldnames = [f.replace('\ufeff', '') for f in reader.fieldnames]

    for row in reader:
        try:
            cursor.execute("""
                INSERT INTO ventes (
                    N_ventes, Adresse_ventes, Date_mutation, Valeur_fonciere,
                    N_dispositions, N_plans, premier_lot, Section, Surface_Carrez_1_lot, 
                    Nb_lots, Type_local, Surface_batie, Nb_pieces_principales, Nature_culture,
                    Nature_culture_speciale, Surface_terrain
                ) VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
                ON DUPLICATE KEY UPDATE
                    Adresse_ventes = VALUES(Adresse_ventes),
                    Date_mutation = VALUES(Date_mutation),
                    Valeur_fonciere = VALUES(Valeur_fonciere),
                    N_dispositions = VALUES(N_dispositions),
                    N_plans = VALUES(N_plans),
                    premier_lot = VALUES(premier_lot),
                    Section = VALUES(Section),
                    Surface_Carrez_1_lot = VALUES(Surface_Carrez_1_lot),
                    Nb_lots = VALUES(Nb_lots),
                    Type_local = VALUES(Type_local),
                    Surface_batie = VALUES(Surface_batie),
                    Nb_pieces_principales = VALUES(Nb_pieces_principales),
                    Nature_culture = VALUES(Nature_culture),
                    Nature_culture_speciale = VALUES(Nature_culture_speciale),
                    Surface_terrain = VALUES(Surface_terrain)
            """, (
                row['N_ventes'],
                row['Adresse_ventes'],
                convert_date(row['Date mutation']),  # ✅ conversion ici
                to_float_or_none(row['Valeur fonciere']),
                to_int_or_none(row['No disposition']),
                row['No plan'],
                row['1er lot'],
                row['Section'],
                to_float_or_none(row['Surface Carrez du 1er lot']),
                to_int_or_none(row['Nombre de lots']),
                row['Type local'],
                to_int_or_none(row['Surface reelle bati']),
                to_int_or_none(row['Nombre pieces principales']),
                row['Nature culture'],
                row['Nature culture speciale'],
                to_int_or_none(row['Surface terrain'])
            ))
        except Exception as e:
            print("Erreur sur la ligne :", row)
            print("Message :", e)

# Commit des insertions
conn.commit()
cursor.close()
conn.close()
print("Import terminé.")

Table regions

In [1]:
import mysql.connector
import csv
from datetime import datetime

# Fonction pour convertir les dates CSV (JJ/MM/AAAA) en format MySQL (AAAA-MM-JJ)
def convert_date(date_str):
    if not date_str or not date_str.strip():
        return None  # si la cellule est vide
    try:
        return datetime.strptime(date_str, "%d/%m/%Y").date().isoformat()
    except ValueError:
        return None  # en cas d'erreur, on met NULL
        
def to_int_or_none(value):
    if not value or value.strip() == "":
        return None
    return int(float(value))  # float() pour gérer "164.87" puis int() si tu veux un entier

def to_float_or_none(value):
    if not value or value.strip() == "":
        return None
    return float(value.replace(",", "."))  # au cas où ton CSV a des virgules


# Connexion à MySQL
conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="",
    database="valeurs_fonciere_v2"
)
cursor = conn.cursor()

# Chemin du fichier CSV
csv_path = r"C:\Users\victo\OneDrive\Bureau\OCP5REFRECH\Tables\Tables_csv\Table_region.csv"

with open(csv_path, mode='r', encoding='utf-8') as file:
    reader = csv.DictReader(file, delimiter=',')
    
    # Supprimer le BOM si présent
    reader.fieldnames = [f.replace('\ufeff', '') for f in reader.fieldnames]

    for row in reader:
        try:
            # Conversion de la géolocalisation en POINT MySQL
            if row['geolocalisation']:
                lat, lon = map(float, row['geolocalisation'].split(','))
                point_wkt = f'POINT({lon} {lat})'
            else:
                point_wkt = None

            cursor.execute("""
                INSERT INTO region (
                    com_code_regions, regrgp_nom, reg_nom, aca_nom,
                    dep_nom, com_nom, ze_id, dep_code, aca_code, 
                    reg_code, AUC_id, AUC_nom, Geolocalisation
                ) VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, ST_GeomFromText(%s))
                ON DUPLICATE KEY UPDATE
                    regrgp_nom = VALUES(regrgp_nom),
                    reg_nom = VALUES(reg_nom),
                    aca_nom = VALUES(aca_nom),
                    dep_nom = VALUES(dep_nom),
                    com_nom = VALUES(com_nom),
                    ze_id = VALUES(ze_id),
                    dep_code = VALUES(dep_code),
                    aca_code = VALUES(aca_code),
                    reg_code = VALUES(reg_code),
                    AUC_id = VALUES(AUC_id),
                    AUC_nom = VALUES(AUC_nom),
                    Geolocalisation = ST_GeomFromText(%s)
            """, (
                row['com_code_regions'],
                row['regrgp_nom'],
                row['reg_nom'],
                row['aca_nom'],
                row['dep_nom'],
                row['com_nom'],
                row['ze_id'],
                row['dep_code'],
                row['aca_code'],
                row['reg_code'],
                row['auc_id'],
                row['auc_nom'],
                point_wkt,  # pour INSERT
                point_wkt   # pour UPDATE
            ))
        except Exception as e:
            print("Erreur sur la ligne :", row)
            print("Message :", e)

# Commit des insertions
conn.commit()
cursor.close()
conn.close()
print("Import terminé.")



Import terminé.


Table commune

In [15]:
import mysql.connector
import csv
from datetime import datetime

# Fonction pour convertir les dates CSV (JJ/MM/AAAA) en format MySQL (AAAA-MM-JJ)
def convert_date(date_str):
    if not date_str or not date_str.strip():
        return None  # si la cellule est vide
    try:
        return datetime.strptime(date_str, "%d/%m/%Y").date().isoformat()
    except ValueError:
        return None  # en cas d'erreur, on met NULL
        
def to_int_or_none(value):
    if not value or value.strip() == "":
        return None
    return int(float(value))  # float() pour gérer "164.87" puis int() si tu veux un entier

def to_float_or_none(value):
    if not value or value.strip() == "":
        return None
    return float(value.replace(",", "."))  # au cas où ton CSV a des virgules


# Connexion à MySQL
conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="Winteriscoming1!",
    database="valeurs_fonciere_v2"
)
cursor = conn.cursor()

# Chemin du fichier CSV
# Chemin du fichier CSV
csv_path = r"C:\Users\victo\OneDrive\Bureau\OCP5REFRECH\Tables\Tables_csv\Table_commune.csv"

with open(csv_path, mode='r', encoding='utf-8') as file:
    reader = csv.DictReader(file, delimiter=',')
    
    # Supprimer le BOM si présent
    reader.fieldnames = [f.replace('\ufeff', '') for f in reader.fieldnames]

    for row in reader:
        try:
            cursor.execute("""
                INSERT INTO communes (
                    cod_dep_com_communes, com_code_commune, COM, PMUN,
                    PCAP, PTOT
                ) VALUES (%s, %s, %s, %s, %s, %s)
                ON DUPLICATE KEY UPDATE
                    cod_dep_com_communes = VALUES(cod_dep_com_communes),
                    com_code_commune = VALUES(com_code_commune),
                    COM = VALUES(COM),
                    PMUN = VALUES(PMUN),
                    PCAP = VALUES(PCAP),
                    PTOT = VALUES(PTOT)
            """, (
                row['COD_DEP_COM'],
                row['CODCOM'],
                row['COM'],
                row['PMUN'],
                row['PCAP'],
                row['PTOT']
            ))
        except Exception as e:
            print("Erreur sur la ligne :", row)
            print("Message :", e)

# Commit des insertions
conn.commit()
cursor.close()
conn.close()
print("Import terminé.")



Import terminé.


Table adress


In [5]:
import mysql.connector
import csv
from datetime import datetime

# Fonction pour convertir les dates CSV (JJ/MM/AAAA) en format MySQL (AAAA-MM-JJ)
def convert_date(date_str):
    if not date_str or not date_str.strip():
        return None  # si la cellule est vide
    try:
        return datetime.strptime(date_str, "%d/%m/%Y").date().isoformat()
    except ValueError:
        return None  # en cas d'erreur, on met NULL
        
def to_int_or_none(value):
    if not value or value.strip() == "":
        return None
    return int(float(value))  # float() pour gérer "164.87" puis int() si tu veux un entier

def to_float_or_none(value):
    if not value or value.strip() == "":
        return None
    return float(value.replace(",", "."))  # au cas où ton CSV a des virgules


# Connexion à MySQL
conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="Winteriscoming1!",
    database="valeurs_fonciere_v2"
)
cursor = conn.cursor()

# Chemin du fichier CSV
csv_path = r"C:\Users\victo\OneDrive\Bureau\OCP5REFRECH\Tables\Tables_csv\Table_adresse.csv"

def to_int_or_none(value):
    if not value or value.strip() == "":
        return None
    return int(value)

with open(csv_path, mode='r', encoding='utf-8') as file:
    reader = csv.DictReader(file, delimiter=',')

    # Supprimer le BOM si présent
    reader.fieldnames = [f.replace('\ufeff', '') for f in reader.fieldnames]

    for row in reader:
        try:
            cursor.execute("""
                INSERT INTO adress (
                    Adresse, Commune, N_Voie, B_T_Q,
                    Type_de_voie, Code_voie, Voie, Code_postal, com_code_adresse, code_dep_commune_adresse, Id_adresse
                ) VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
                ON DUPLICATE KEY UPDATE
                    Adresse = VALUES(Adresse),
                    Commune = VALUES(Commune),
                    N_Voie = VALUES(N_Voie),
                    B_T_Q = VALUES(B_T_Q),
                    Type_de_voie = VALUES(Type_de_voie),
                    Code_voie = VALUES(Code_voie),
                    Voie = VALUES(Voie),
                    Code_postal = VALUES(Code_postal),
                    com_code_adresse = VALUES(com_code_adresse),
                    code_dep_commune_adresse = VALUES(code_dep_commune_adresse),
                    Id_adresse = VALUES(Id_adresse)
            """, (
                row['Adresse'],
                row['Commune'],
                to_int_or_none(row['No voie']),  # ✅ conversion pour éviter les chaînes vides
                row['B/T/Q'] if row['B/T/Q'].strip() != "" else None,
                row['Type de voie'] if row['Type de voie'].strip() != "" else None,
                row['Code voie'] if row['Code voie'].strip() != "" else None,
                row['Voie'] if row['Voie'].strip() != "" else None,
                row['Code postal'] if row['Code postal'].strip() != "" else None,
                row['Code commune'] if row['Code commune'].strip() != "" else None,
                row['Code_dep_commune_adresse'] if row['Code_dep_commune_adresse'].strip() != "" else None,
                to_int_or_none(row['Id_adresse'])
            ))
        except Exception as e:
            print("Erreur sur la ligne :", row)
            print("Message :", e)

# Commit des insertions
conn.commit()
cursor.close()
conn.close()
print("Import terminé.")

Import terminé.


Table vente

In [12]:
import mysql.connector
import csv
from datetime import datetime

# Fonction pour convertir les dates CSV (JJ/MM/AAAA) en format MySQL (AAAA-MM-JJ)
def convert_date(date_str):
    if not date_str or not date_str.strip():
        return None  # si la cellule est vide
    try:
        return datetime.strptime(date_str, "%d/%m/%Y").date().isoformat()
    except ValueError:
        return None  # en cas d'erreur, on met NULL
        
def to_int_or_none(value):
    if not value or value.strip() == "":
        return None
    return int(float(value))  # float() pour gérer "164.87" puis int() si tu veux un entier

def to_float_or_none(value):
    if not value or value.strip() == "":
        return None
    return float(value.replace(",", "."))  # au cas où ton CSV a des virgules


# Connexion à MySQL
conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="Winteriscoming1!",
    database="valeurs_fonciere_v2"
)
cursor = conn.cursor()

# Chemin du fichier CSV
csv_path = r"C:\Users\victo\OneDrive\Bureau\OCP5REFRECH\Tables\Tables_csv\Table_ventes.csv"

with open(csv_path, mode='r', encoding='utf-8') as file:
    reader = csv.DictReader(file, delimiter=',')
    
    # Supprimer le BOM si présent
    reader.fieldnames = [f.replace('\ufeff', '') for f in reader.fieldnames]

    for row in reader:
        try:
            cursor.execute("""
                INSERT INTO ventes (
                    N_ventes, Date_mutation, Valeur_fonciere,
                    N_dispositions, N_plans, premier_lot, Section, Surface_Carrez_1_lot, 
                    Nb_lots, Type_local, Surface_batie, Nb_pieces_principales, Nature_culture,
                    Nature_culture_speciale, Surface_terrain, Id_adresse_ventes
                ) VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
                ON DUPLICATE KEY UPDATE
                    N_ventes = VALUES(N_ventes),
                    Date_mutation = VALUES(Date_mutation),
                    Valeur_fonciere = VALUES(Valeur_fonciere),
                    N_dispositions = VALUES(N_dispositions),
                    N_plans = VALUES(N_plans),
                    premier_lot = VALUES(premier_lot),
                    Section = VALUES(Section),
                    Surface_Carrez_1_lot = VALUES(Surface_Carrez_1_lot),
                    Nb_lots = VALUES(Nb_lots),
                    Type_local = VALUES(Type_local),
                    Surface_batie = VALUES(Surface_batie),
                    Nb_pieces_principales = VALUES(Nb_pieces_principales),
                    Nature_culture = VALUES(Nature_culture),
                    Nature_culture_speciale = VALUES(Nature_culture_speciale),
                    Surface_terrain = VALUES(Surface_terrain),
                    Id_adresse_ventes = VALUES(Id_adresse_ventes)
            """, (
                row['N_ventes'],
                convert_date(row['Date mutation']),  # ✅ conversion ici
                to_float_or_none(row['Valeur fonciere']),
                to_int_or_none(row['No disposition']),
                row['No plan'],
                row['1er lot'],
                row['Section'],
                to_float_or_none(row['Surface Carrez du 1er lot']),
                to_int_or_none(row['Nombre de lots']),
                row['Type local'],
                to_int_or_none(row['Surface reelle bati']),
                to_int_or_none(row['Nombre pieces principales']),
                row['Nature culture'],
                row['Nature culture speciale'],
                to_int_or_none(row['Surface terrain']),
                to_int_or_none(row['Id_adresse_ventes'])
            ))
        except Exception as e:
            print("Erreur sur la ligne :", row)
            print("Message :", e)

# Commit des insertions
conn.commit()
cursor.close()
conn.close()
print("Import terminé.")

Import terminé.
